In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

import gc

In [ ]:
data = np.load("user_pca_reduced_and_counts.npz", allow_pickle=True)

In [ ]:
user_pca_reduced = data["user_pca_reduced"].item()

In [ ]:
embeddings_raw = np.stack(list(user_pca_reduced.values()))
usernames = list(user_pca_reduced.keys())

In [ ]:
pca_dim= 20

In [ ]:
k= 25

In [ ]:
embeddings = embeddings_raw[:, :pca_dim]

In [ ]:
mbk = MiniBatchKMeans(
    n_clusters=k,
    batch_size=100000,
    n_init=3,
    random_state=42
)

In [ ]:
cluster_ids = mbk.fit_predict(embeddings)

In [ ]:
user_cluster_dict = dict(zip(usernames, cluster_ids))

In [ ]:
user_cluster_dict

{'BitcoinBellyB': np.int32(19),
 'BTCMEXGlobal': np.int32(17),
 'CryptoNetDaily': np.int32(23),
 'Bitstamp': np.int32(18),
 'btcusdalerts': np.int32(12),
 'BitcoinAverage': np.int32(12),
 'K_9Girl': np.int32(15),
 'BTCtoCAD': np.int32(12),
 'WSJ': np.int32(18),
 'coindata_uk': np.int32(12),
 'fixtheblock': np.int32(3),
 'awesomepersona2': np.int32(18),
 'JeremyKissel': np.int32(23),
 'cryptowat_ch': np.int32(12),
 'vpaul8483': np.int32(21),
 'littleprinceJD': np.int32(3),
 'abubakarhayus': np.int32(17),
 'RealTradeAlts': np.int32(12),
 'zloadr': np.int32(3),
 'amneesiack': np.int32(9),
 'crypto_buy_tips': np.int32(3),
 'StephenPKendal': np.int32(12),
 'VolumeMonitor': np.int32(14),
 'XCryptoRTaeP': np.int32(15),
 'bmurphypointman': np.int32(3),
 'Jimmmyz169': np.int32(22),
 'nic__carter': np.int32(18),
 'heatrockz': np.int32(1),
 'JacobCanfield': np.int32(18),
 'WallStSenpai': np.int32(3),
 'ledgerstatus': np.int32(1),
 'yackzilon': np.int32(2),
 'bot_blockchain': np.int32(18),
 'ccinf

In [ ]:
cluster_centers = mbk.cluster_centers_
top_n = 10
cluster_top_users = {}  # cluster_id -> list of usernames

for cid in range(k):
    # User in diesem Cluster
    idx_in_cluster = np.where(cluster_ids == cid)[0]
    cluster_embs = embeddings[idx_in_cluster]

    # Distanz zum Centroid
    distances = np.linalg.norm(cluster_embs - cluster_centers[cid], axis=1)

    # 10 nächste User
    nearest_idx = idx_in_cluster[np.argsort(distances)[:top_n]]
    cluster_top_users[cid] = [usernames[i] for i in nearest_idx]

In [ ]:
cluster_top_users

{0: ['Hattak_Han',
  'bocaintommy',
  'TrumpSmart2',
  'SaschaDre',
  'Dhara_MSD',
  'Elvishonz',
  'nha1331',
  'Genisys_P1',
  'LEETEABAGGER69',
  'Estefan86653779'],
 1: ['ColoradoHodl',
  'satoshiscards',
  'btc_sensei',
  'btcDiamondHands',
  '42069Blockchain',
  'zachdoteth',
  'unrealreddit',
  'marmolmaroto',
  'JREDBOKIE',
  'mangobulll'],
 2: ['GRenteriaGard',
  'sebastianpozzo',
  'nacholibreYo',
  'karbonboy',
  'BenGomez83',
  'ZipperZbieracz',
  'sofianeouafir',
  'PaterickStar',
  'SAnthonyH',
  'rakshitagarwall'],
 3: ['CrptoSheikh',
  'BuddyLMusic',
  'Tonywenlambo',
  'PAWZ_Official',
  'WhaleStreet101',
  '0xaMidas',
  'Lucrativeflip',
  'SEOMalkamDior',
  'OmbithoSignal',
  'ccw991'],
 4: ['i_goziem',
  'AMCRYPTONERD',
  'TheRealOQ',
  'jack48586041',
  'Dogetot92096307',
  'ismail_Rseven',
  'CryptoKing_SKN',
  'SnoopCrypto1',
  'DenizTokat16',
  'KING_BOOZEY'],
 5: ['oogachi_',
  'Alfusu1',
  'TornikeXozreva2',
  'RByonk',
  'AdyudityaChita',
  'LD94690675',
  'La

In [ ]:
import joblib

joblib.dump(mbk, "kmeans_20d_25k.pkl")

['/content/drive/MyDrive/kmeans_20d_25k.pkl']

In [ ]:
np.savez_compressed(
    "user_cluster_dict_and_cluster_top_users.npz",
    user_cluster_dict=user_cluster_dict,
    cluster_top_users=cluster_top_users
)